# Homework: Tenerife RAG + Tools
por: Enmanuel Alejandro De Oleo

### Objetivo Principal:
Desarrollar un prototipo conversacional reproducible que combine búsqueda semántica,
mantenimiento de contexto y llamadas a servicios externos, todo desde un único
notebook.

1. **RAG sobre `inputs/TENERIFE.pdf`**: una guía con lugares de interés de la isla (zonas, rutas, playas, miradores, pueblos, contiene fotos... pero para este proyecto no lo utilizaremos), expuesta como una herramienta de búsqueda documental (`search_tenerife_guide`).
2. **Herramienta de clima** (`weather.py`): consulta el pronóstico de [WeatherAPI](https://www.weatherapi.com/) para Tenerife u otra ciudad (`get_weather`).

En lugar de un RAG con recuperación fija, declaramos ambas capacidades como herramientas (*tool calling*) y dejamos que el modelo decida cuándo necesita buscar en la guía, cuándo necesita consultar el tiempo, o cuándo puede responder directamente.

In [ ]:
# Importante correr esta linea para instalar todas las dependencias del proyecto
%pip install -q -U openai python-dotenv pypdf numpy faiss-cpu jsonschema tiktoken


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 0. Configuración e importación de funciones que necesitaremos.

Cargamos credenciales desde `.env` (`OPENAI_API_KEY` para generación, embeddings y tool calling) y definimos los modelos generativo y de embeddings, los parámetros de generación expuestos (`MODEL_PARAMS`) y la ruta al PDF `data/TENERIFE.pdf`.

In [75]:
import faiss
import getpass
import json
import logging
import os
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import numpy as np
import tiktoken
from dotenv import load_dotenv
from jsonschema import Draft202012Validator
from openai import OpenAI
from pypdf import PdfReader

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

client = OpenAI()

# Profesor tuve que utilizar mi propia API KEY y este modelo, por algun motivo me dejo de funcionar la API Key Suya. Entonces seguí adelante con esta configuración.
GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-5.4-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

PDF_PATH = Path("inputs/TENERIFE.pdf")

print("Modelo generativo (tool calling):", GENERATION_MODEL)
print("Modelo de embeddings (RAG):", EMBEDDING_MODEL)
print("PDF:", PDF_PATH.resolve())

Modelo generativo (tool calling): gpt-5.4-mini
Modelo de embeddings (RAG): text-embedding-3-small
PDF: /Users/enmanueldeoleo/Pontia/pontia-llm-homework/inputs/TENERIFE.pdf


In [53]:
MODEL_PARAMS = {
    "temperature": 0.2,
    "top_p": 1.0,
    "max_output_tokens": 600,
}

# Tool Calling (Para fines de experimentación antes de la implementación)
De la misma forma en como practicamos en la sessión 4, voy a hacer una pequeña implementación prueba de  mi función que trae el clima de Tenerife para los siguientes 3 dias.

## Primera interacción entre la LLM y mi Function Call
Como veremos a continuación, intentaremos generar intereacción entre el LLM de OPEN IA con mi funcion que devuelve el clima en TENERIFE. Este es un ensayo, más adelante utilizaremos funciones que nos ayuden a mejorar esta interacción. Pero como buena práctica, estamos asegurando que funcione de manera correcta antes de seguir avanzando.

La API devuelve bastante información para esta practica solo estamos tomando los siguientes valores:
`date`, `max_temp`, `min_temp`, `condition`

In [107]:
from weather import Weather

weather_client = Weather()
weather_tool_schema = weather_client.get_weather_tool_schema()

print(weather_client.get_weather())
print(weather_tool_schema)

[{'date': '2026-06-16', 'max_temp': 20.9, 'min_temp': 19.2, 'condition': 'Sunny'}, {'date': '2026-06-17', 'max_temp': 20.9, 'min_temp': 19.5, 'condition': 'Sunny'}, {'date': '2026-06-18', 'max_temp': 20.7, 'min_temp': 19.2, 'condition': 'Sunny'}]
{'type': 'function', 'name': 'get_weather', 'description': "Retorna el clima actual y de los proximos 3 días de Tenerife. Por defecto tiene el valor de 'Tenerife' y 3 días de pronostico. Puede recibir otro parametro 'q' para buscar otra ciudad.", 'parameters': {'type': 'object', 'properties': {'q': {'type': 'string', 'description': "Nombre de la ciudad del que se quiere obtener el clima. Por Ejemplo: 'Tenerife', 'Madrid', 'London'"}, 'days_from_now': {'type': 'integer', 'description': 'Número de días desde hoy del que se quiere obtener el clima. Por Ejemplo: 1, 2, 3'}}, 'required': ['q', 'days_from_now'], 'additionalProperties': False}}


In [108]:
question = "¿Cómo estará el clima hoy en Tenerife?"

response = client.responses.create(
    model=GENERATION_MODEL,
    input=question,
    tools=[weather_tool_schema],
)

for item in response.output:
    print("TYPE:", item.type)
    print(item)
    print()

TYPE: function_call
ResponseFunctionToolCall(arguments='{"q":"Tenerife","days_from_now":0}', call_id='call_YxkSzM8FzsB1B4bisXQTb6sc', name='get_weather', type='function_call', id='fc_030a3a67fb04f609006a3101b6b07c81a380700cf0b596c951', namespace=None, status='completed')



In [109]:
function_calls = [item for item in response.output if item.type == "function_call"]

if not function_calls:
    print("El modelo no ha solicitado ninguna herramienta.")
else:
    call = function_calls[0]
    args = json.loads(call.arguments)
    print("Herramienta solicitada:", call.name)
    print("Argumentos:", args)

    if call.name == "get_weather":
        tool_result = weather_client.get_weather(**args)
    else:
        raise ValueError(f"Herramienta no soportada: {call.name}")

    print("Resultado de la herramienta:")
    print(json.dumps(tool_result, indent=2, ensure_ascii=False))

Herramienta solicitada: get_weather
Argumentos: {'q': 'Tenerife', 'days_from_now': 0}
Resultado de la herramienta:
[
  {
    "date": "2026-06-16",
    "max_temp": 20.9,
    "min_temp": 19.2,
    "condition": "Sunny"
  }
]


In [110]:
if function_calls:
    final_response = client.responses.create(
        model=GENERATION_MODEL,
        input=[
            {"role": "user", "content": question},
            call,
            {
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(tool_result, ensure_ascii=False),
            },
        ],
        tools=[weather_tool_schema],
    )

    print(final_response.output_text)

Hoy en Tenerife estará **soleado**.  
Temperatura estimada: **máxima 20.9 °C** y **mínima 19.2 °C**.


## 1. Carga y chunking de la guía de Tenerife

Seguimos el mismo patrón que la sesión 4: cargamos el PDF con `pypdf` (una entrada por página con texto plano y número de página) y lo troceamos con `chunk_text`, una función de partición manual por caracteres con solape. A cada chunk le añadimos `page` y `chunk_index` en la metadata para poder citar la fuente más adelante.

In [111]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {PDF_PATH.resolve()}")

reader = PdfReader(str(PDF_PATH))
pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    pages.append({"page": i, "text": text})

print("Número de páginas cargadas:", len(pages))
print("Primeros caracteres de la página 0:\n")
print(pages[0]["text"][:500])

Número de páginas cargadas: 25
Primeros caracteres de la página 0:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo 
dirección Pl


In [112]:
def chunk_text(text: str, *, chunk_size: int = 900, overlap: int = 150) -> list[str]:
    """Divide texto en chunks con solape por caracteres, evitando bucles infinitos."""
    cleaned = "\n".join(line.strip() for line in text.splitlines() if line.strip())
    if len(cleaned) <= chunk_size:
        return [cleaned]

    chunks = []
    start = 0
    while start < len(cleaned):
        end = min(start + chunk_size, len(cleaned))
        chunks.append(cleaned[start:end])
        if end == len(cleaned):
            break
        start = max(0, end - overlap)
    return chunks


chunks: list[dict[str, Any]] = []
for page_data in pages:
    for i, chunk in enumerate(chunk_text(page_data["text"])):
        chunks.append(
            {
                "id": f"page_{page_data['page']}::chunk_{i}",
                "source": PDF_PATH.name,
                "page": page_data["page"],
                "chunk_index": i,
                "text": chunk,
            }
        )

# Por alguna extraña razon, me estaba generando un chunk[6]["text"] vacío y me estaba rompiendo todo... entonces tuve que poner este guard para que solo me considere los chucks con texto.
chunks = [c for c in chunks if c["text"]]

print("Número de chunks:", len(chunks))
print("\nEjemplo de chunk:\n")
print(chunks[6]["text"])
print("\nMetadata:", {k: v for k, v in chunks[0].items() if k != "text"})

Número de chunks: 31

Ejemplo de chunk:

o Playa de Benijo [vídeo - ubicación]
Playa de arena negra, si el día está despejado, de los atardeceres más
espectaculares que podéis ver en la isla.
Para llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede
desde aquí (puede ser que os encontréis con el acceso cortado con una valla,
pero da igual, podéis acceder igualmente).
• La Orotava
Este es el pueblo más bonito y maravilloso del mundo (no es porque yo sea de allí).

Metadata: {'id': 'page_0::chunk_0', 'source': 'TENERIFE.pdf', 'page': 0, 'chunk_index': 0}


## 2. Embeddings y vector store

Generamos embeddings con `client.embeddings.create` (modelo `text-embedding-3-small` de OpenAI) y los indexamos en un `faiss.IndexFlatIP`. Normalizamos los vectores antes de añadirlos al índice para que el producto interior equivalga a similitud coseno.

In [113]:
def embed_texts(texts: list[str], *, model: str = EMBEDDING_MODEL) -> np.ndarray:
    """Genera embeddings para una lista de textos con la API de OpenAI."""
    response = client.embeddings.create(model=model, input=texts)
    vectors = [item.embedding for item in response.data]
    return np.array(vectors, dtype=np.float32)


chunk_texts_list = [chunk["text"] for chunk in chunks]
chunk_embeddings = embed_texts(chunk_texts_list)

# Normalizar antes de añadir al índice para que el producto interior == similitud coseno
faiss.normalize_L2(chunk_embeddings)
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print("Shape embeddings:", chunk_embeddings.shape)
print("Vectores en el índice FAISS:", index.ntotal)
print("Primeros IDs:", [c["id"] for c in chunks[:3]])

Shape embeddings: (31, 1536)
Vectores en el índice FAISS: 31
Primeros IDs: ['page_0::chunk_0', 'page_1::chunk_0', 'page_2::chunk_0']


In [114]:
def search_tenerife_docs(query: str, *, k: int = 3) -> list[dict[str, Any]]:
    """Busca chunks relevantes usando FAISS (similitud coseno vía producto interior normalizado)."""
    query_embedding = embed_texts([query])
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk = chunks[int(idx)]
        results.append(
            {
                "rank": rank,
                "score": float(score),
                "source": chunk["source"],
                "page": chunk["page"],
                "chunk_index": chunk["chunk_index"],
                "text": chunk["text"],
            }
        )
    return results


def mostrar_resultados(results: list[dict[str, Any]]) -> None:
    for r in results:
        print(f"Resultado {r['rank']} | {r['source']} página {r['page']} chunk {r['chunk_index']} | score={r['score']:.3f}")
        print(r["text"][:300])
        print("-" * 80)


sample_results = search_tenerife_docs("¿Qué se puede ver en Santa Cruz de Tenerife?", k=3)
mostrar_resultados(sample_results)

Resultado 1 | TENERIFE.pdf página 0 chunk 0 | score=0.722
TENERIFE – LUGARES DE INTERÉS
SITIOS QUE VER
ZONA NORTE
• Santa Cruz de Tenerife:
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa
Cruz sería:
- Aparcar en el aparcamiento del Parque Marítimo (ubicación).
- Caminar por la Avenida Marítima hasta Plaza de España
--------------------------------------------------------------------------------
Resultado 2 | TENERIFE.pdf página 10 chunk 0 | score=0.596
tendréis poca playa (especialmente a la del Ancón) y tened mucho respeto al mar en
estas playas, que la corriente es muy traicionera.
• Puerto de La Cruz
Es la zona turística del Norte de Tenerife. De hecho, hace años era el núcleo turístico
de la isla hasta que empezó a ganar mucho peso el Sur de T
--------------------------------------------------------------------------------
Resultado 3 | TENERIFE.pdf página 2 chunk 0 | score=0.574
o Parque García Sanabria [vídeo - ubicación]
o Playa de Las T

## 3. Herramienta RAG: `search_tenerife_guide`

Envolvemos `vector_store.similarity_search` en una función que devuelve un contexto compacto con fuentes (página y `chunk_id`, como en la sesión 3), y definimos el esquema que verá el modelo. La descripción deja claro cuándo debe usarse esta herramienta y cuándo no (preguntas sobre el clima).

In [115]:
def format_docs_with_sources(results: list[dict[str, Any]]) -> str:
    """Convierte resultados de búsqueda en un bloque de contexto con fuentes."""
    blocks = []
    for r in results:
        blocks.append(
            f"[Fuente: {r['source']}, página {r['page']}, chunk {r['chunk_index']}]\n{r['text']}"
        )
    return "\n\n---\n\n".join(blocks)


def search_tenerife_guide_tool(query: str, k: int) -> dict[str, Any]:
    """Busca información en la guía de lugares de interés de Tenerife."""
    safe_k = max(1, min(k, 8))
    results = search_tenerife_docs(query, k=safe_k)
    return {
        "query": query,
        "context": format_docs_with_sources(results),
    }


tenerife_rag_tool_schema = {
    "type": "function",
    "name": "search_tenerife_guide",
    "description": (
        "Busca fragmentos relevantes en la guía de lugares de interés de Tenerife "
        "(zonas, rutas, playas, miradores, pueblos y actividades). Úsala para preguntas "
        "sobre qué ver o hacer en Tenerife. No la uses para preguntas sobre el clima para esto tendremos otra función."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Pregunta o búsqueda semántica sobre lugares, zonas o actividades en Tenerife.",
            },
            "k": {
                "type": "integer",
                "description": "Número de fragmentos a recuperar. Usa 3 o 4 salvo que necesites más cobertura.",
                "minimum": 1,
                "maximum": 8,
            },
        },
        "required": ["query", "k"],
        "additionalProperties": False,
    },
}

## 4. Logging de los intentos de *function call*

Configuramos un logger dedicado (`tenerife_rag.tools`) que escribe en consola y en `tool_calls.log`, con marca de tiempo, nombre de la herramienta, argumentos, resultado (`ok`/`error`) y duración. El ejecutor de la siguiente sección registra ahí cada intento.

In [116]:
LOG_PATH = Path("tool_calls.log")

tool_logger = logging.getLogger("tenerife_rag.tools")
tool_logger.setLevel(logging.INFO)
tool_logger.handlers.clear()
tool_logger.propagate = False

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
file_handler.setFormatter(formatter)
tool_logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)
tool_logger.addHandler(console_handler)

print("Log de herramientas en:", LOG_PATH.resolve())

Log de herramientas en: /Users/enmanueldeoleo/Pontia/pontia-llm-homework/tool_calls.log


## 5. Ejecutor de herramientas

Registro de herramientas (`ToolSpec`), validación de argumentos con `jsonschema`, ejecución con traza (`ToolExecution`) registrada en `tool_logger`, y un bucle (`run_llm_with_tools`) que llama al modelo con `MODEL_PARAMS`, ejecuta las herramientas solicitadas y vuelve a llamar al modelo hasta obtener una respuesta final.

In [117]:
@dataclass
class ToolSpec:
    """Contrato completo de una herramienta disponible para el modelo."""

    schema: dict[str, Any]
    function: Callable[..., Any]

    @property
    def name(self) -> str:
        return self.schema["name"]

    @property
    def parameters_schema(self) -> dict[str, Any]:
        return self.schema.get("parameters", {"type": "object", "properties": {}})


@dataclass
class ToolExecution:
    """Registro observable de una llamada real a herramienta."""

    name: str
    arguments: dict[str, Any]
    ok: bool
    output: Any
    elapsed_seconds: float


@dataclass
class ToolRunResult:
    """Resultado final de ejecutar un prompt con herramientas."""

    final_response: Any
    conversation: list[Any]
    executions: list[ToolExecution] = field(default_factory=list)

    @property
    def output_text(self) -> str:
        return self.final_response.output_text

    @property
    def tool_names(self) -> list[str]:
        return [execution.name for execution in self.executions]


def validate_tool_arguments(tool: ToolSpec, arguments: dict[str, Any]) -> None:
    """Valida argumentos usando el JSON Schema declarado para la herramienta."""
    validator = Draft202012Validator(tool.parameters_schema)
    errors = sorted(validator.iter_errors(arguments), key=lambda error: error.path)
    if errors:
        messages = [error.message for error in errors]
        raise ValueError("Argumentos inválidos: " + "; ".join(messages))


def execute_tool_call(call: Any, registry: dict[str, ToolSpec]) -> tuple[dict[str, Any], ToolExecution]:
    """Ejecuta una llamada de herramienta del modelo, registra el intento en el log y devuelve la traza."""
    start = time.perf_counter()
    name = getattr(call, "name", "")

    try:
        arguments = json.loads(call.arguments or "{}")
        if name not in registry:
            raise ValueError(f"Herramienta no registrada: {name}")

        tool = registry[name]
        validate_tool_arguments(tool, arguments)
        output = tool.function(**arguments)
        ok = True
        payload = {"ok": ok, "data": output}
    except Exception as exc:
        arguments = locals().get("arguments", {})
        output = {"error_type": type(exc).__name__, "message": str(exc)}
        ok = False
        payload = {"ok": ok, "error": output}

    elapsed = time.perf_counter() - start
    execution = ToolExecution(name=name, arguments=arguments, ok=ok, output=output, elapsed_seconds=elapsed)

    if ok:
        tool_logger.info("tool_call name=%s args=%s ok=%s elapsed=%.3fs", name, arguments, ok, elapsed)
    else:
        tool_logger.warning(
            "tool_call name=%s args=%s ok=%s error=%s elapsed=%.3fs", name, arguments, ok, output, elapsed
        )

    return payload, execution


def run_llm_with_tools(
    user_input: str,
    *,
    tools: list[ToolSpec],
    instructions: str | None = None,
    model: str = GENERATION_MODEL,
    model_params: dict[str, Any] = MODEL_PARAMS,
    max_tool_rounds: int = 5,
    tool_choice: str | dict[str, Any] = "auto",
    parallel_tool_calls: bool = True,
) -> ToolRunResult:
    """Ejecuta un bucle de tool calling hasta obtener una respuesta final."""
    registry = {tool.name: tool for tool in tools}
    tool_schemas = [tool.schema for tool in tools]
    conversation: list[Any] = [{"role": "user", "content": user_input}]
    executions: list[ToolExecution] = []

    for round_index in range(max_tool_rounds):
        current_tool_choice = tool_choice if round_index == 0 else "auto"
        response = client.responses.create(
            model=model,
            instructions=instructions,
            input=conversation,
            tools=tool_schemas,
            tool_choice=current_tool_choice,
            parallel_tool_calls=parallel_tool_calls,
            temperature=model_params["temperature"],
            top_p=model_params["top_p"],
            max_output_tokens=model_params["max_output_tokens"],
        )

        function_calls = [item for item in response.output if item.type == "function_call"]
        if not function_calls:
            return ToolRunResult(final_response=response, conversation=conversation, executions=executions)

        conversation.extend(response.output)
        for call in function_calls:
            payload, execution = execute_tool_call(call, registry)
            executions.append(execution)
            conversation.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(payload, ensure_ascii=False),
                }
            )

    raise RuntimeError(f"Se alcanzó el límite de {max_tool_rounds} rondas de herramientas.")

## 6. Herramienta de clima (`weather.py`)

Reutilizamos directamente la clase `Weather` del proyecto: `get_weather` es la función real que se ejecuta, y `get_weather_tool_schema()` es el esquema que ve el modelo. Por defecto consulta el clima de Tenerife para los próximos días.

In [118]:
weather_tool = ToolSpec(schema=weather_tool_schema, function=weather_client.get_weather)
print(json.dumps(weather_tool_schema, indent=2, ensure_ascii=False))


{
  "type": "function",
  "name": "get_weather",
  "description": "Retorna el clima actual y de los proximos 3 días de Tenerife. Por defecto tiene el valor de 'Tenerife' y 3 días de pronostico. Puede recibir otro parametro 'q' para buscar otra ciudad.",
  "parameters": {
    "type": "object",
    "properties": {
      "q": {
        "type": "string",
        "description": "Nombre de la ciudad del que se quiere obtener el clima. Por Ejemplo: 'Tenerife', 'Madrid', 'London'"
      },
      "days_from_now": {
        "type": "integer",
        "description": "Número de días desde hoy del que se quiere obtener el clima. Por Ejemplo: 1, 2, 3"
      }
    },
    "required": [
      "q",
      "days_from_now"
    ],
    "additionalProperties": false
  }
}


## 7. Asistente con ambas herramientas

Combinamos la herramienta de búsqueda documental (`search_tenerife_guide`) y la herramienta de clima (`get_weather`) en una sola lista de herramientas, con instrucciones que indican cuándo usar cada una. El modelo decide, para cada pregunta, si necesita buscar en la guía, consultar el tiempo, ambas cosas o ninguna.

In [130]:
tenerife_rag_tool = ToolSpec(schema=tenerife_rag_tool_schema, function=search_tenerife_guide_tool)

ALL_TOOLS = [tenerife_rag_tool, weather_tool]

TENERIFE_ASSISTANT_INSTRUCTIONS = (
    "Eres un asistente que ayuda a planear visitas a Tenerife. "
    "Usa search_tenerife_guide para preguntas sobre lugares, zonas, rutas, playas o actividades en la isla. "
    "Usa get_weather para preguntas sobre el tiempo actual o el pronóstico de los próximos días. "
    "No uses herramientas si la pregunta puede responderse de forma general sin datos externos. " #Si la pregunta es generica, tal vez no necesitamos utilizar las tools.
    "Si search_tenerife_guide no devuelve información suficiente, dilo claramente en vez de inventar lugares. " # Evitar alucinar.
    "Cuando respondas a partir de la guía, indica la página de la que sale la información. " # Lo de las referencias
    "Ten en cuenta el historial de la conversación para resolver referencias como 'allí' o 'esa zona'." # Contextualizar mejor.
)

### 7.1. Pregunta sobre lugares (RAG)

In [131]:
rag_run = run_llm_with_tools(
    "¿Qué lugares de interés puedo visitar en la zona norte de Tenerife?",
    instructions=TENERIFE_ASSISTANT_INSTRUCTIONS,
    tools=ALL_TOOLS,
)

print(rag_run.output_text)
print("\nHerramientas usadas:", rag_run.tool_names)


2026-06-16 10:32:16,888 | INFO | tool_call name=search_tenerife_guide args={'query': 'zona norte de Tenerife lugares de interés qué visitar pueblos miradores playas rutas actividades', 'k': 4} ok=True elapsed=0.916s


En la **zona norte de Tenerife** la guía destaca sobre todo estos lugares de interés:

- **Santa Cruz de Tenerife**: la capital. Recomienda pasear por la **Avenida Marítima**, ver la **Plaza de España**, el **Auditorio de Tenerife**, subir por la **Calle Castillo** hacia **Plaza Weyler**, pasar por el **Parque García Sanabria** y volver por la **Plaza del Príncipe**. También sugiere ir a la **playa de Las Teresitas**.  
  **Fuente:** *TENERIFE.pdf, página 0*

- **Parque Marítimo César Manrique**: conjunto de piscinas naturales en Santa Cruz.  
  **Fuente:** *TENERIFE.pdf, página 0*

- **El Teide**: aunque no está “en la costa norte” como tal, la guía lo incluye como visita imprescindible desde el norte. Recomienda subir por la **TF-24 (carretera de La Esperanza)** y parar en el **Mirador de La Tarta**; también menciona el **Mirador de Chipeque** para ver el mar de nubes.  
  **Fuente:** *TENERIFE.pdf, página 14*

- **Centro de Visitantes de El Portillo**: útil si quieres más informació

### 7.2. Pregunta sobre el clima (`weather.py`)

In [132]:
weather_run = run_llm_with_tools(
    "¿Qué tiempo va a hacer en Tenerife durante los próximos 3 días?",
    instructions=TENERIFE_ASSISTANT_INSTRUCTIONS,
    tools=ALL_TOOLS,
)

print(weather_run.output_text)
print("\nHerramientas usadas:", weather_run.tool_names)
for execution in weather_run.executions:
    print(json.dumps(execution.output, indent=2, ensure_ascii=False, default=str))


2026-06-16 10:32:32,161 | INFO | tool_call name=get_weather args={'q': 'Tenerife', 'days_from_now': 3} ok=True elapsed=0.116s


En Tenerife, para los próximos 3 días se espera tiempo soleado:

- 16/06/2026: máxima 20.9 °C, mínima 19.2 °C, soleado
- 17/06/2026: máxima 20.9 °C, mínima 19.5 °C, soleado
- 18/06/2026: máxima 20.7 °C, mínima 19.2 °C, soleado

Si quieres, también te puedo decir qué zona de la isla suele ser más agradable según ese tiempo.

Herramientas usadas: ['get_weather']
[
  {
    "date": "2026-06-16",
    "max_temp": 20.9,
    "min_temp": 19.2,
    "condition": "Sunny"
  },
  {
    "date": "2026-06-17",
    "max_temp": 20.9,
    "min_temp": 19.5,
    "condition": "Sunny"
  },
  {
    "date": "2026-06-18",
    "max_temp": 20.7,
    "min_temp": 19.2,
    "condition": "Sunny"
  }
]


### 7.3. Pregunta combinada: lugar + clima (Scenario un poco mas complejo)

In [133]:
combined_run = run_llm_with_tools(
    "Quiero ir a la playa de Las Teresitas pasado mañana. ¿Qué tiempo hará y qué más puedo ver por la zona?",
    instructions=TENERIFE_ASSISTANT_INSTRUCTIONS,
    tools=ALL_TOOLS,
)

print(combined_run.output_text)
print("\nHerramientas usadas:", combined_run.tool_names)
for execution in combined_run.executions:
    print(execution.name, execution.arguments, "ok=", execution.ok)


2026-06-16 10:32:42,488 | INFO | tool_call name=get_weather args={'q': 'Tenerife', 'days_from_now': 2} ok=True elapsed=0.103s
2026-06-16 10:32:42,664 | INFO | tool_call name=search_tenerife_guide args={'query': 'playa de Las Teresitas y qué ver por la zona cerca de Santa Cruz de Tenerife', 'k': 4} ok=True elapsed=0.175s


Pasado mañana en Tenerife se espera **sol** y una temperatura de alrededor de **20.9 °C de máxima** y **19.5 °C de mínima**.

Sobre la zona de **Las Teresitas / Santa Cruz**, la guía recomienda este plan:
- **Parque Marítimo César Manrique**: conjunto de piscinas naturales.
- **Avenida Marítima** hasta **Plaza de España**.
- Ver el **Auditorio de Tenerife** por el camino.
- Pasear por el centro: **Calle Castillo**, **Plaza Weyler**, **Parque García Sanabria** y **Plaza del Príncipe**.

Fuente: **TENERIFE.pdf, página 0**.

Si quieres, también te puedo proponer un **plan de medio día** combinando playa + paseo por Santa Cruz.

Herramientas usadas: ['get_weather', 'search_tenerife_guide']
get_weather {'q': 'Tenerife', 'days_from_now': 2} ok= True
search_tenerife_guide {'query': 'playa de Las Teresitas y qué ver por la zona cerca de Santa Cruz de Tenerife', 'k': 4} ok= True


### 7.4. Pregunta general, sin necesidad de herramientas

Este caso es importante porque no deberia buscar dentro RAG si es una pregunta general que el LLM puede contestar por si mismo.

In [135]:
general_run = run_llm_with_tools(
    "Explícame en una frase qué es un plan de viaje.",
    instructions=TENERIFE_ASSISTANT_INSTRUCTIONS,
    tools=ALL_TOOLS,
)

print(general_run.output_text)
print("\nHerramientas usadas:", general_run.tool_names)


Un plan de viaje es una organización previa de destinos, fechas, actividades y logística para aprovechar mejor un viaje.

Herramientas usadas: []


## 8. Memoria multiturno y control de tokens

`ConversationManager` guarda el historial de la conversación (incluyendo *function calls* y sus salidas) y, antes de cada turno, recorta los mensajes más antiguos si el historial supera `max_tokens` (usando `tiktoken` para contar). `run_chat_turn` añade el mensaje del usuario a la memoria, llama al modelo con `MODEL_PARAMS` y ejecuta las herramientas que pida, guardando toda la traza en la conversación.

In [136]:
def count_tokens(messages: list[Any], *, model: str = GENERATION_MODEL) -> int:
    """Cuenta tokens aproximados de una lista de mensajes/elementos de conversación."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("o200k_base")

    text = "\n".join(str(message) for message in messages)
    return len(encoding.encode(text))


class ConversationManager:
    """Mantiene el historial de una conversación multiturno y controla su longitud en tokens."""

    def __init__(self, *, max_tokens: int = 3000, model: str = GENERATION_MODEL):
        self.max_tokens = max_tokens
        self.model = model
        self.history: list[Any] = []

    def add(self, message: Any) -> None:
        self.history.append(message)
        self._trim()

    def extend(self, messages: list[Any]) -> None:
        self.history.extend(messages)
        self._trim()

    def _trim(self) -> None:
        """Elimina los mensajes más antiguos mientras se supere el presupuesto de tokens."""
        while len(self.history) > 1 and count_tokens(self.history, model=self.model) > self.max_tokens:
            removed = self.history.pop(0)
            tool_logger.info("Historial recortado (> %d tokens), eliminado: %s", self.max_tokens, str(removed)[:120])

    @property
    def messages(self) -> list[Any]:
        return self.history

In [137]:
def run_chat_turn(
    conversation: ConversationManager,
    user_input: str,
    *,
    tools: list[ToolSpec],
    instructions: str | None = None,
    model: str = GENERATION_MODEL,
    model_params: dict[str, Any] = MODEL_PARAMS,
    max_tool_rounds: int = 5,
) -> str:
    """Procesa un turno de usuario: actualiza memoria, llama al modelo y ejecuta herramientas si las pide."""
    registry = {tool.name: tool for tool in tools}
    tool_schemas = [tool.schema for tool in tools]

    conversation.add({"role": "user", "content": user_input})

    for _ in range(max_tool_rounds):
        response = client.responses.create(
            model=model,
            instructions=instructions,
            input=conversation.messages,
            tools=tool_schemas,
            tool_choice="auto",
            temperature=model_params["temperature"],
            top_p=model_params["top_p"],
            max_output_tokens=model_params["max_output_tokens"],
        )

        function_calls = [item for item in response.output if item.type == "function_call"]
        conversation.extend(response.output)

        if not function_calls:
            return response.output_text

        for call in function_calls:
            payload, _execution = execute_tool_call(call, registry)
            conversation.add(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(payload, ensure_ascii=False),
                }
            )

    raise RuntimeError(f"Se alcanzó el límite de {max_tool_rounds} rondas de herramientas.")

### 8.1. Demo: conversación multiturno con memoria, tokens y logging

In [138]:
conversation = ConversationManager(max_tokens=3000, model=GENERATION_MODEL)

turn_1 = run_chat_turn(
    conversation,
    "¿Qué lugares de interés hay en la zona norte de Tenerife?",
    tools=ALL_TOOLS,
    instructions=TENERIFE_ASSISTANT_INSTRUCTIONS,
)
print("Turno 1:\n", turn_1)

2026-06-16 10:35:06,262 | INFO | tool_call name=search_tenerife_guide args={'query': 'zona norte de Tenerife lugares de interés qué ver pueblos miradores playas rutas actividades', 'k': 4} ok=True elapsed=1.434s


Turno 1:
 En la **zona norte de Tenerife** la guía destaca varios lugares de interés, sobre todo en torno a **Santa Cruz, La Laguna, el Teide y el norte costero**. Según la guía:

- **Santa Cruz de Tenerife**: paseo por la **Avenida Marítima**, **Plaza de España**, **Auditorio de Tenerife**, **Calle Castillo**, **Plaza Weyler**, **Parque García Sanabria** y **Plaza del Príncipe**.  
- **Parque Marítimo César Manrique**: conjunto de piscinas naturales.  
- **Playa de Las Teresitas**: una de las playas más conocidas de la zona.  
- **Subida al Teide por la TF-24 (carretera de La Esperanza)**: con parada en el **Mirador de La Tarta**; también menciona el **Mirador de Chipeque** para ver el mar de nubes.  
- **Parque Nacional del Teide**: la guía lo considera imprescindible; sugiere el **Parador de las Cañadas** y el **mirador de La Ruleta**.  
- **Centro de Visitantes de El Portillo**: recomendado si quieres más información sobre el Teide.

Además, en la parte de la guía sobre el norte ap

In [127]:
turn_2 = run_chat_turn(
    conversation,
    "¿Y en la zona sur? Dame dos opciones.",
    tools=ALL_TOOLS,
    instructions=TENERIFE_ASSISTANT_INSTRUCTIONS,
)
print("Turno 2:\n", turn_2)

2026-06-16 09:57:01,322 | INFO | tool_call name=search_tenerife_guide args={'query': 'zona sur de Tenerife lugares de interés dos opciones playas actividades pueblos miradores', 'k': 4} ok=True elapsed=0.163s
2026-06-16 09:57:03,561 | INFO | Historial recortado (> 3000 tokens), eliminado: {'role': 'user', 'content': '¿Qué lugares de interés hay en la zona norte de Tenerife?'}
2026-06-16 09:57:03,563 | INFO | Historial recortado (> 3000 tokens), eliminado: ResponseFunctionToolCall(arguments='{"query":"zona norte de Tenerife lugares de interés qué ver pueblos miradores playas


Turno 2:
 Sí, en la **zona sur** la guía destaca estas dos opciones:

1. **Costa Adeje**  
   Es la zona turística por excelencia, con hoteles de lujo, ocio y playas de arena blanca.  
   - Playas recomendadas: **Playa de Torviscas** y **Playa de Fañabé**.  
   - **Fuente:** *TENERIFE.pdf, página 15*

2. **El Médano / La Tejita**  
   Son playas muy típicas de Tenerife, con la imagen de **Montaña Roja** al fondo. La guía avisa de que **La Tejita es nudista**.  
   - **Fuente:** *TENERIFE.pdf, página 22*

Si quieres, te puedo dar ahora **dos opciones más del sur pero centradas en pueblos o miradores**, no solo playas.


In [139]:
turn_3 = run_chat_turn(
    conversation,
    "De esas opciones de la zona sur, ¿qué tiempo hará allí los próximos 2 días?",
    tools=ALL_TOOLS,
    instructions=TENERIFE_ASSISTANT_INSTRUCTIONS,
)
print("Turno 3:\n", turn_3)

print("\nMensajes en memoria:", len(conversation.messages))
print("Tokens aproximados en memoria:", count_tokens(conversation.messages, model=GENERATION_MODEL))

2026-06-16 10:35:21,709 | INFO | tool_call name=get_weather args={'q': 'Tenerife', 'days_from_now': 2} ok=True elapsed=0.089s


Turno 3:
 Para **Tenerife** en los próximos **2 días** se espera:

- **2026-06-16**: **Soleado**, máxima **20.9 °C**, mínima **19.2 °C**
- **2026-06-17**: **Soleado**, máxima **20.9 °C**, mínima **19.5 °C**

Si quieres, también te puedo decir **qué zona sur encaja mejor con ese tiempo** para playa o paseo.

Mensajes en memoria: 8
Tokens aproximados en memoria: 2140


## 9. Streaming de la respuesta final (Bonus de la tarea segun del PDF.)

Versión streaming de `run_chat_turn`: pasa `stream=True` a la API para recibir el texto token a token conforme lo genera el modelo. Los turnos intermedios donde el modelo pide herramientas no producen texto visible, así que el streaming solo se aprecia en el turno final de cada pregunta. (en la segunda clase vimos un ejemplo de esto)

In [ ]:
import sys


def run_chat_turn_streaming(
    conversation: ConversationManager,
    user_input: str,
    *,
    tools: list[ToolSpec],
    instructions: str | None = None,
    model: str = GENERATION_MODEL,
    model_params: dict[str, Any] = MODEL_PARAMS,
    max_tool_rounds: int = 5,
) -> str:
    """Como run_chat_turn pero hace streaming del texto de la respuesta final."""
    registry = {tool.name: tool for tool in tools}
    tool_schemas = [tool.schema for tool in tools]

    conversation.add({"role": "user", "content": user_input})

    for _ in range(max_tool_rounds):
        stream = client.responses.create(
            model=model,
            instructions=instructions,
            input=conversation.messages,
            tools=tool_schemas,
            tool_choice="auto",
            stream=True, # tal cual lo hicimos en clase.
            temperature=model_params["temperature"],
            top_p=model_params["top_p"],
            max_output_tokens=model_params["max_output_tokens"],
        )

        final_response = None
        for event in stream:
            if event.type == "response.output_text.delta":
                sys.stdout.write(event.delta)
                sys.stdout.flush()
            elif event.type == "response.completed":
                final_response = event.response

        if final_response is None:
            raise RuntimeError("No se recibió respuesta final del stream.")

        function_calls = [item for item in final_response.output if item.type == "function_call"]
        conversation.extend(final_response.output)

        if not function_calls:
            return final_response.output_text

        for call in function_calls:
            payload, _execution = execute_tool_call(call, registry)
            conversation.add({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(payload, ensure_ascii=False),
            })

    raise RuntimeError(f"Se alcanzó el límite de {max_tool_rounds} rondas de herramientas.")

In [ ]:
stream_conversation = ConversationManager(max_tokens=3000, model=GENERATION_MODEL)

print("Respuesta en streaming:")

run_chat_turn_streaming(
    stream_conversation,
    "¿Qué lugares puedo visitar en la zona sur de Tenerife? Dame un resumen.",
    tools=ALL_TOOLS,
    instructions=TENERIFE_ASSISTANT_INSTRUCTIONS,
)

Respuesta en streaming:


2026-06-16 09:51:23,395 | INFO | tool_call name=search_tenerife_guide args={'query': 'zona sur de Tenerife lugares para visitar resumen playas miradores pueblos actividades', 'k': 4} ok=True elapsed=0.184s


Claro. En la **zona sur de Tenerife** la guía destaca sobre todo estas opciones:

- **Costa Adeje**: es la zona turística por excelencia, con muchos hoteles, ocio y playas de arena blanca.  
  - Playas recomendadas: **Playa de Torviscas** y, en general, las playas de la zona.  
  - También es buena zona si buscas ambiente, restaurantes y actividades de ocio.  
  - **Fuente: TENERIFE.pdf, página 15**

- **El Médano / La Tejita**: playas muy típicas de Tenerife, con la imagen de **Montaña Roja** al fondo.  
  - **La Tejita** es además **playa nudista**.  
  - **Fuente: TENERIFE.pdf, página 22**

- **Los Cristianos**: la guía recomienda sobre todo la **Playa de Los Cristianos** y la **Playa de Las Vistas**.  
  - Es una zona cómoda para pasear y disfrutar de playa.  
  - **Fuente: TENERIFE.pdf, página 22**

**Resumen rápido:** si buscas **playa y ambiente turístico**, ve a **Costa Adeje**; si prefieres un paisaje más icónico y de playa abierta, **El Médano / La Tejita**; y si quieres una 

'Claro. En la **zona sur de Tenerife** la guía destaca sobre todo estas opciones:\n\n- **Costa Adeje**: es la zona turística por excelencia, con muchos hoteles, ocio y playas de arena blanca.  \n  - Playas recomendadas: **Playa de Torviscas** y, en general, las playas de la zona.  \n  - También es buena zona si buscas ambiente, restaurantes y actividades de ocio.  \n  - **Fuente: TENERIFE.pdf, página 15**\n\n- **El Médano / La Tejita**: playas muy típicas de Tenerife, con la imagen de **Montaña Roja** al fondo.  \n  - **La Tejita** es además **playa nudista**.  \n  - **Fuente: TENERIFE.pdf, página 22**\n\n- **Los Cristianos**: la guía recomienda sobre todo la **Playa de Los Cristianos** y la **Playa de Las Vistas**.  \n  - Es una zona cómoda para pasear y disfrutar de playa.  \n  - **Fuente: TENERIFE.pdf, página 22**\n\n**Resumen rápido:** si buscas **playa y ambiente turístico**, ve a **Costa Adeje**; si prefieres un paisaje más icónico y de playa abierta, **El Médano / La Tejita**; y

## 10. Recapitulación

En este notebook hemos:

1. Conectado con un LLM comercial (OpenAI, Responses API) con parámetros de generación expuestos y configurables vía `.env` (`MODEL_PARAMS`: `temperature`, `top_p`, `max_output_tokens`).
2. Cargado `inputs/TENERIFE.pdf` con `PyPDFLoader`, troceado con `RecursiveCharacterTextSplitter` (igual que en la sesión 3) e indexado los chunks con `GoogleGenerativeAIEmbeddings` en un `InMemoryVectorStore`, recuperando y citando la página y el `chunk_id` de origen.
3. Expuesto esa búsqueda como herramienta (`search_tenerife_guide`) y reutilizado `weather.py` tal cual (`get_weather` / `get_weather_tool_schema`) para la herramienta de clima, sin modificar el módulo.
4. Construido un ejecutor de herramientas con validación de argumentos (`jsonschema`), manejo de errores y logging de cada intento (correcto o fallido) en consola y en `tool_calls.log`.
5. Combinado ambas herramientas en un asistente (sección 7) que decide, pregunta a pregunta, qué herramienta usar.
6. Añadido memoria multiturno con control de longitud en tokens (`ConversationManager` + `tiktoken`) y un bucle de chat (`run_chat_turn`) que resuelve referencias al historial ("allí", "esa zona") a lo largo de varios turnos (sección 8).

11. Reflexiones finales
- Creo que ha sido una muy buena oportunidad de aprendizaje, y gracias maestro por los notebooks tan completos de las clases anteriores, realmente no hubo demasiado misterio para poder hacer este ejercicio.